# 06 — Direct Preference Optimization (DPO) Training
**Goal**: Train the SFT model using Direct Preference Optimization (DPO) on execution debug trajectory pairs ($ \beta=0.1 $, LR $5\times 10^{-5}$). Produces `./checkpoints/dpo/final` for RQ5 comparison against PPO.

---

## Step 1: Environment Setup & Universal Path Resolution

In [ ]:
!pip install -q trl peft bitsandbytes accelerate datasets transformers marimo tyro

import sys, os, shutil, importlib

# Universal Multi-Platform Path Resolution (Marimo / MoLab / Colab / Kaggle / Local)
def prepare_environment_src():
    curr = os.path.abspath(os.getcwd())
    if os.path.exists(os.path.join(curr, 'src', 'models', 'loader.py')):
        print(f"Using local 'src' directory at {curr}")
        return curr
    
    for colab_dir in ['/content/self-correction-llm-rl', '/content']:
        if os.path.exists(os.path.join(colab_dir, 'src', 'models', 'loader.py')):
            print(f"Using MoLab / Colab directory at {colab_dir}")
            return colab_dir
    
    if os.path.exists('/kaggle/input'):
        working_src = '/kaggle/working/src'
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'models' in dirs and os.path.exists(os.path.join(root, 'models', 'loader.py')):
                if os.path.exists(working_src):
                    shutil.rmtree(working_src)
                shutil.copytree(root, working_src)
                print(f"Copied 'src' from {root} to {working_src}")
                return '/kaggle/working'
            elif 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'models', 'loader.py')):
                src_dir = os.path.join(root, 'src')
                if os.path.exists(working_src):
                    shutil.rmtree(working_src)
                shutil.copytree(src_dir, working_src)
                print(f"Copied 'src' from {src_dir} to {working_src}")
                return '/kaggle/working'
    
    parent = os.path.abspath('..')
    if os.path.exists(os.path.join(parent, 'src')):
        return parent
    return curr

repo_root = prepare_environment_src()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path added: {repo_root}")

print("Using repository src/training/dpo.py; no in-notebook source patching.")

for mod in ['src.training.dpo', 'src.training']:
    if mod in sys.modules:
        del sys.modules[mod]

import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.training.dpo import make_preference_pairs, run_dpo_training

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")


## Step 2: Collect Preference Trajectory Pairs `(prompt, chosen, rejected)`
Runs $K=3$ debug rollouts on APPS problems. Extracts winning solutions (AC) as `chosen` and failing solutions (CE/RE/WA) as `rejected`.

In [ ]:
!pip uninstall -y torchao

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
SFT_CHECKPOINT = "./checkpoints/sft/final"

search_paths = [
    "./checkpoints/sft/final",
    "/content/checkpoints/sft/final",
    "/content/self-correction-llm-rl/checkpoints/sft/final",
    "/content/drive/MyDrive/checkpoints/sft/final",
    "/kaggle/working/checkpoints/sft/final",
    "/kaggle/input"
]

model_path = None
for p in search_paths:
    if os.path.exists(p):
        if os.path.exists(os.path.join(p, 'adapter_config.json')):
            model_path = p
            break
        for root, dirs, files in os.walk(p):
            if 'adapter_config.json' in files and 'sft' in root.lower():
                model_path = root
                break
    if model_path:
        break

if not model_path:
    raise FileNotFoundError("SFT adapter checkpoint is required for DPO; refusing base-model fallback.")

print(f"Loading model from {model_path} for DPO pair collection in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(model_name=model_path, load_in_4bit=False, lora_r=16)

print("\nLoading APPS dataset for preference pair generation via Parquet branch...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split='train[:50]')
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)

print("Generating preference dataset...")
preference_data = make_preference_pairs(apps_clean, model, tokenizer, K=3)
print(f"Total DPO preference pairs generated: {len(preference_data)}")
if len(preference_data) == 0:
    raise ValueError("No execution-grounded DPO preference pairs were generated; refusing to train on an empty or reference-fallback dataset.")


## Step 3: Run DPO Training
Uses PEFT reference model trick to avoid loading duplicate reference weights in VRAM.

In [ ]:
import sys, importlib
if 'src.training.dpo' in sys.modules:
    del sys.modules['src.training.dpo']
import src.training.dpo
importlib.reload(src.training.dpo)
from src.training.dpo import run_dpo_training

print("Starting DPO Training...")
dpo_trainer = run_dpo_training(
    model=model,
    tokenizer=tokenizer,
    preference_data=preference_data,
    output_dir="./checkpoints/dpo",
    beta=0.1,
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
)

print("\nDPO Training completed successfully!")
final_dir="./checkpoints/dpo/final"
with open(os.path.join(final_dir,"dpo_metadata.json"),"w",encoding="utf-8") as f:
    json.dump({"checkpoint_type":"dpo_execution_grounded","preference_source":"model_generated_execution_rollouts","reference_solution_fallback":False},f,indent=2)
print("Saved final DPO adapter checkpoint to ./checkpoints/dpo/final")


## Step 4: Checkpoint Verification & Inference Test
Verifies saved DPO adapter files, reloads weights, confirms LoRA config, and executes 3 APPS inference tests.

In [ ]:
import os
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint_dir = None
search_roots = ['./checkpoints/dpo/final', '/content/checkpoints/dpo/final', '/content/self-correction-llm-rl/checkpoints/dpo/final', '/kaggle/working/checkpoints/dpo/final', '/kaggle/working', '/kaggle/input']

for root in search_roots:
    if os.path.exists(root):
        if os.path.exists(os.path.join(root, 'adapter_config.json')):
            checkpoint_dir = root
            break
        for r, dirs, files in os.walk(root):
            if 'adapter_config.json' in files and 'dpo' in r.lower():
                checkpoint_dir = r
                break
    if checkpoint_dir:
        break

if not checkpoint_dir:
    checkpoint_dir = "./checkpoints/dpo/final"

print(f"=== 1. Inspecting DPO Checkpoint Files in {checkpoint_dir} ===")
if os.path.exists(checkpoint_dir):
    for fname in sorted(os.listdir(checkpoint_dir)):
        fpath = os.path.join(checkpoint_dir, fname)
        if os.path.isfile(fpath):
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"  - {fname}: {size_mb:.2f} MB")

    print("\n=== 2. Verifying DPO LoRA Configuration ===")
    config = PeftConfig.from_pretrained(checkpoint_dir)
    print(f"  - lora_alpha: {getattr(config, 'lora_alpha', 'N/A')}")
    print(f"  - r: {getattr(config, 'r', 'N/A')}")
    print(f"  - target_modules: {list(getattr(config, 'target_modules', []))}")
    print(f"  - peft_type: {getattr(config, 'peft_type', 'N/A')}")

    print("\n=== 3. Reloading Saved DPO Model & Adapter ===")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    reloaded_dpo = PeftModel.from_pretrained(base_model, checkpoint_dir)
    reloaded_dpo.eval()
    print("Reloaded DPO adapter model successfully!")

    print("\n=== 4. Running APPS Inference Check (3 Examples) ===")
    sample_prompts = [
        apps_clean[0]['question'],
        apps_clean[1]['question'],
        apps_clean[2]['question']
    ]

    for idx, p in enumerate(sample_prompts):
        prompt_text = f"### Problem:\n{p[:300]}\n\n### Solution:\n```python\n"
        inputs = tokenizer(prompt_text, return_tensors="pt").to(next(reloaded_dpo.parameters()).device)
        with torch.no_grad():
            out = reloaded_dpo.generate(**inputs, max_new_tokens=100, do_sample=False)
        gen_text = tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"\n--- DPO Inference Sample {idx + 1} ---")
        print(gen_text[:250] + "...")

    print("\nCheckpoint verification completed successfully: adapter files, LoRA configuration, model reload, and inference generation verified.")
